# Z2 vs sigma in 3D — dependence on the bond cutoff R and on Anderson disorder W

L=10, repulsion kappa_shift = 3 r0, MJ=2, 100 realizations.
- `01_z2_vs_sigma_cutoff_R/`: W=0, R = 1.05, 1.1, 1.15, 1.2, 1.25, 1.3 (r0)
- `02_z2_vs_sigma_anderson_W/`: R=1.3, W = 2, 4, 6, 8, 10, 12
- Reference R=1.3, W=0: the R=1.3 run of `01_z2_vs_sigma_cutoff_R/` (recomputed, see README)

Copy the `results_*` folders from the cluster into the corresponding subfolder before running.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import cmasher as csh

HERE = Path.cwd()
FSS_DIR = HERE.parents[1] / "appendix" / "finite_size_scaling"
sys.path.append(str(FSS_DIR))
from load_files_from_cluster import load_cluster_results

matplotlib.rcParams.update(
    {
        "font.size": 10,
        "text.usetex": True,
        "font.family": "serif",
        "font.serif": ["Times"],
    }
)
aps_figwidth = 6.6

In [ ]:
# ── run parameters (must match the submitted jobs) ───────────
L = 10
KSHIFT = 3.0
MJ = 2.0
SIGMA_RANGE = "0.0-0.25"
R_REF, W_REF = 1.3, 0.0

R_VALUES = [1.05, 1.1, 1.15, 1.2, 1.25, 1.3]
W_VALUES = [2.0, 4.0, 6.0, 8.0, 10.0, 12.0]


def results_dir(folder, R, W):
    return HERE / folder / (
        f"results_3d_L{L}_kshift{KSHIFT}r0_R{R}_W{W}_MJ{MJ}_sigma{SIGMA_RANGE}"
    )


def load_or_none(path, pattern):
    # skip runs that are not copied back yet instead of crashing the notebook
    try:
        return load_cluster_results(path, pattern)
    except FileNotFoundError as e:
        print(f"!! missing: {e}")
        return None


def check_attrs(out, **expected):
    # make sure we are plotting what we think we are plotting
    attrs = out["attrs"]
    for key, val in expected.items():
        assert np.isclose(attrs[key], val), f"{key}={attrs[key]} != {val}"

In [ ]:
# ── analysis 1: cutoff R, W=0 (includes the reference R=1.3) ──
out_R = {}
for R in R_VALUES:
    out = load_or_none(results_dir("01_z2_vs_sigma_cutoff_R", R, 0.0), "sigma_*.h5")
    if out is not None:
        check_attrs(out, L=L, W=0.0, MJ=MJ, kappa_shift_r0=KSHIFT, R_cutoff=R)
        out_R[R] = out

# ── analysis 2: Anderson W, R=1.3 ─────────────────────────────
out_W = {}
if R_REF in out_R:
    out_W[W_REF] = out_R[R_REF]
for W in W_VALUES:
    out = load_or_none(results_dir("02_z2_vs_sigma_anderson_W", R_REF, W), "sigma_*.h5")
    if out is not None:
        check_attrs(out, L=L, W=W, MJ=MJ, kappa_shift_r0=KSHIFT, R_cutoff=R_REF)
        out_W[W] = out

In [ ]:
def curve(out):
    # sigma in units of r0, <z2> and its standard error over realizations
    sigma_r0 = out["sigma_values"] * L
    z2 = np.asarray(out["z2"], dtype=float)
    mean = z2.mean(axis=1)
    sem = z2.std(axis=1, ddof=1) / np.sqrt(z2.shape[1])
    return sigma_r0, mean, sem


for name, data in [("R", out_R), ("W", out_W)]:
    for key, out in sorted(data.items()):
        n_sig, n_real = np.shape(out["z2"])
        print(f"{name}={key}: {n_sig} sigma values x {n_real} realizations")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(aps_figwidth, 2.6), sharey=True, dpi=350)
cmap = csh.bubblegum


def plot_family(axis, data, label_fmt):
    keys = sorted(data)
    for i, key in enumerate(keys):
        color = cmap(i / max(len(keys) - 1, 1))
        s, m, e = curve(data[key])
        axis.plot(s, m, color=color, linewidth=0.9, label=label_fmt(key), rasterized=True)
        axis.fill_between(s, m - e, m + e, color=color, alpha=0.25, linewidth=0, rasterized=True)


plot_family(ax[0], out_R, lambda R: rf"$R={R:g}\,r_0$")
plot_family(ax[1], out_W, lambda W: rf"$W={W:g}$")

ax[0].set_ylabel(r"$\langle \nu_{\textup{3D}} \rangle$")
ax[0].set_xlabel(r"$\sigma\ (r_0)$")
ax[1].set_xlabel(r"$\sigma\ (r_0)$")
for axis in ax:
    axis.set_xlim(0, 0.25)
    axis.legend(fontsize=8, frameon=False, loc="lower right")
ax[0].set_xticks([0, 0.125, 0.25])
ax[0].set_xticklabels(["0", "0.125", ""])  # avoid overlap with the right panel
ax[1].set_xticks([0, 0.125, 0.25])
ax[1].set_xticklabels(["", "0.125", "0.25"])
ax[0].set_ylim(-1.05, 1.05)
ax[0].set_yticks([-1, 0, 1])
fig.subplots_adjust(wspace=0.05)
ax[0].set_title(r"$L=10$, $\rho=3r_0$, $R$ scan ($W=0$)")
ax[1].set_title(r"$W$ scan ($R=1.3\,r_0$)")
fig.savefig(HERE / "fig_z2_vs_sigma_3D.png", dpi=350, bbox_inches="tight")